1. Object Detection with YOLOv5 (Using Pretrained Weights)
Problem: Detect objects using pre-trained YOLOv5 model.

In [1]:
from ultralytics import YOLO

# Load YOLOv8n pretrained on COCO
model = YOLO("yolov8n.pt")

# Run inference
results = model("bus.jpg")
results[0].show()


image 1/1 C:\Users\keert\Desktop\AI\Deep Learning\Assignment\2.Deep_Learning_Question_Set2\4.Coding\bus.jpg: 448x640 1 bus, 366.2ms
Speed: 76.0ms preprocess, 366.2ms inference, 18.0ms postprocess per image at shape (1, 3, 448, 640)


2. Preprocessing Image for ResNet Input
Problem: Write a function to preprocess image for pretrained ResNet.

In [2]:
from torchvision import transforms
from PIL import Image
import torch

def preprocess_image_for_resnet(image_path):
    preprocess = transforms.Compose([
        transforms.Resize(256),              # resize shorter side to 256
        transforms.CenterCrop(224),          # crop to 224x224
        transforms.ToTensor(),               # convert to tensor [0,1]
        transforms.Normalize(                # normalize with ImageNet mean/std
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])
    
    # Load image
    img = Image.open(image_path).convert("RGB")
    img_tensor = preprocess(img).unsqueeze(0)  # add batch dimension
    return img_tensor


3. CNN for CIFAR-10 Classification
Problem: Build a CNN model to classify CIFAR-10 images.

In [3]:
import tensorflow as tf
from tensorflow.keras import layers, models

# 1. Load CIFAR-10 dataset
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

# Normalize pixel values to [0,1]
x_train, x_test = x_train / 255.0, x_test / 255.0

# 2. Build CNN model
model = models.Sequential([
    # Convolutional Block 1
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(32,32,3)),
    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Dropout(0.25),

    # Convolutional Block 2
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Dropout(0.25),

    # Fully Connected Layers
    layers.Flatten(),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')  # 10 classes
])

# 3. Compile model
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

#Augumentation
datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)
datagen.fit(x_train)  # compute statistics if needed

# 4. Train model
history = model.fit(x_train, y_train,
                    epochs=10,
                    batch_size=64,
                    validation_data=(x_test, y_test))

# 5. Evaluate
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=2)
print(f"Test accuracy: {test_acc:.2f}")

C:\Users\keert\anaconda3\envs\deeplearning\lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 80s 93ms/step - accuracy: 0.4112 - loss: 1.6120 - val_accuracy: 0.5474 - val_loss: 1.2856
Epoch 2/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 71s 90ms/step - accuracy: 0.5758 - loss: 1.1868 - val_accuracy: 0.6401 - val_loss: 1.0076
Epoch 3/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 71s 91ms/step - accuracy: 0.6391 - loss: 1.0288 - val_accuracy: 0.6784 - val_loss: 0.9103
Epoch 4/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 71s 91ms/step - accuracy: 0.6762 - loss: 0.9187 - val_accuracy: 0.7096 - val_loss: 0.8345
Epoch 5/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 73s 94ms/step - accuracy: 0.6998 - loss: 0.8554 - val_accuracy: 0.7260 - val_loss: 0.7975
Epoch 6/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 70s 90ms/step - accuracy: 0.7205 - loss: 0.7954 - val_accuracy: 0.7470 - val_loss: 0.7350
Epoch 7/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 71s 90ms/step - accuracy: 0.7340 - loss: 0.7559 - val_accuracy: 0.7474 - val_loss: 0.7250
Epoch 8/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 71s 90ms/step - accuracy: 0.7485 - loss: 0.7153 - 

4. Face Mask Detection – Real-Time Webcam Classifier
Problem: Build a real-time face mask detector using a trained CNN model and OpenCV.

In [6]:
import cv2
import numpy as np
import tensorflow as tf


model = tf.keras.models.load_model("mask_detection_model.h5")

class_names = ['Mask', 'NoMask']

face_cascade = cv2.CascadeClassifier("haarcascade_frontalface_default.xml")

cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)  

def preprocess_face(face_bgr):
    # Convert BGR (OpenCV default) to RGB
    face_rgb = cv2.cvtColor(face_bgr, cv2.COLOR_BGR2RGB)
    # Resize to model input size
    face_resized = cv2.resize(face_rgb, (150, 150))
    # Normalize
    face_norm = face_resized / 255.0
    return np.expand_dims(face_norm, axis=0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1,
                                          minNeighbors=5, minSize=(60, 60))

    for (x, y, w, h) in faces:
        face = frame[y:y+h, x:x+w]
        if face.size == 0:
            continue

        face_array = preprocess_face(face)
        pred = model.predict(face_array, verbose=0)[0]

        # Handle sigmoid vs softmax
        if len(pred) == 1:  # sigmoid output
            label = "Mask" if pred[0] < 0.5 else "NoMask"
            confidence = pred[0] if label == "NoMask" else 1 - pred[0]
        else:               # softmax output
            label_index = np.argmax(pred)
            label = class_names[label_index]
            confidence = pred[label_index]

        # Draw bounding box and label
        color = (0, 255, 0) if label == "Mask" else (0, 0, 255)
        cv2.rectangle(frame, (x, y), (x+w, y+h), color, 2)
        cv2.putText(frame, f"{label} ({confidence:.2f})", (x, y - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2)

    cv2.imshow("Face Mask Detection", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Scenario: Custom Transfer Learning with Frozen + Trainable Layers
Problem: Load a pretrained MobileNetV2 model, freeze base layers, add custom classifier, and fine-tune the top few layers.

In [5]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model

base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224,224,3))
for layer in base_model.layers[:-20]:
    layer.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
output = Dense(2, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=output)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])